In [1]:
import torch 
import torch.nn
import torchvision

from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter


DATA setup

In [2]:


train_dir = "pizza_steak_sushi/train"
test_dir = "pizza_steak_sushi/test"

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])       

train_data = datasets.ImageFolder(train_dir, transform=manual_transforms)
test_data = datasets.ImageFolder(test_dir, transform=manual_transforms)

class_names = train_data.classes
device = "cuda" if torch.cuda.is_available() else "cpu"
train_dataloader = DataLoader(train_data,  batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_data,  batch_size=32, shuffle=False)


Model setup

In [3]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT # NEW in torchvision 0.13, "DEFAULT" means "best weights available"
model = torchvision.models.efficientnet_b0(weights = weights).to(device)

for param in model.features.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Dropout(p=.2, inplace=True),
    nn.Linear(in_features=1280, out_features=3, bias=True)
).to(device)

from torchinfo import summary
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


summary(model, 
        input_size=(32, 3, 224, 224), # (batch_size, color_channels, height, width)
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)



Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 3]              --                   Partial
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   False
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 

In [4]:
def create_writer(experiment_name:str,
                  model_name:str,
                  extra: str = None):
    from datetime import datetime
    import os 

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    if extra:
        # Create log directory path
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name)
        
    return SummaryWriter(log_dir=log_dir)

In [5]:
from modules.engine import train_step, test_step
from tqdm import tqdm


def train(model: torch.nn.Module, 
          train_dataloader: torch.utils.data.DataLoader, 
          test_dataloader: torch.utils.data.DataLoader, 
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device,
          writer: torch.utils.tensorboard.writer.SummaryWriter) -> dict[str, list]:
    
    results = {
        "train_loss" : [],
        "train_acc" : [],
        "test_loss" : [],
        "test_acc" : []
    }

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model,
                                           train_dataloader,
                                           loss_fn,
                                           optimizer,
                                           device)
        
        print(f"epoch {epoch+1} / {epochs}")
        print(f"Train loss: {train_loss}")
        print(f"Train acc: {train_acc}")
        

        test_loss, test_acc = test_step(model,
                                    test_dataloader,
                                    loss_fn,
                                    device)
        print(f"Test loss: {test_loss}")
        print(f"Test acc: {test_acc}")
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)


        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)

        if writer:
            writer.add_scalars(main_tag="Loss", 
                            tag_scalar_dict={"train_loss": train_loss,
                                                "test_loss": test_loss},
                            global_step=epoch)

            # Add accuracy results to SummaryWriter
            writer.add_scalars(main_tag="Accuracy", 
                            tag_scalar_dict={"train_acc": train_acc,
                                                "test_acc": test_acc}, 
                            global_step=epoch)
            
            # Track the PyTorch model architecture
            writer.add_graph(model=model, 
                            # Pass in an example input
                            input_to_model=torch.randn(32, 3, 224, 224).to(device))
        
            writer.close()
        
        

    return results

c:\Python Latop\razerblade-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def set_seeds(seed:int = 42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

In [9]:
example_writer = create_writer(experiment_name="data_10_percent",
                               model_name="effnetb0",
                               extra="5_epochs")

open: tensorboard --logdir runs

to clear: Remove-Item -Recurse -Force runs
    - then re-initialize writer

In [10]:
set_seeds()
results = train(model=model,
                train_dataloader= train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                epochs=5,
                device=device,
                writer= example_writer)
results

Training: 100%|██████████| 8/8 [00:01<00:00,  4.95it/s]


epoch 1 / 5
Train loss: 0.5206151083111763
Train acc: 0.92578125


Testing: 100%|██████████| 3/3 [00:00<00:00,  5.40it/s]


Test loss: 0.5660428603490194
Test acc: 0.9280303030303031


Training: 100%|██████████| 8/8 [00:01<00:00,  5.28it/s]


...

epoch 5 / 5
Train loss: 0.4891447313129902
Train acc: 0.79296875


Testing: 100%|██████████| 3/3 [00:00<00:00,  5.66it/s]


Test loss: 0.4885110755761464
Test acc: 0.8768939393939394


100%|██████████| 5/5 [00:28<00:00,  5.62s/it]


{'train_loss': [0.5206151083111763,
  0.5439257062971592,
  0.5026287250220776,
  0.450826995074749,
  0.4891447313129902],
 'train_acc': [0.92578125, 0.80078125, 0.9375, 0.93359375, 0.79296875],
 'test_loss': [0.5660428603490194,
  0.530373235543569,
  0.4721509913603465,
  0.4660722315311432,
  0.4885110755761464],
 'test_acc': [0.9280303030303031,
  0.9071969696969697,
  0.90625,
  0.9280303030303031,
  0.8768939393939394]}